# Abstract

- Goal: Code basic Collaborative Filtering and make interactive form for it

- Dataset: [MovieLens 100K Dataset (Kaggle)](https://www.kaggle.com/datasets/prajitdatta/movielens-100k-dataset)

- Project Details:

    I coded a basic Collaborative Filtering algorithm. I use a Similarity matrix for all users for prediction and recommendation. I calculate a similarity matrix with `sklearn.metrics.pairwise.cosine_similarity`.

  
- Best result: 1.0176 (RMSE Score of Collaborative Filtering)

- Sections:
    - [Imports](#Imports)
    - [Utils](#Utils)
    - [Dataset](#Dataset)
    - [Modeling](#Modeling)
    - [Prediction](#Prediction)
        - [Setup Form](#Setup-Form)
        - [Prediction Form](#Prediction-Form)

# Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from itertools import combinations
from IPython.display import display, clear_output
import ipywidgets as widgets

# Utils

In [4]:
def find_movie_by_id(id_, select_col, movie_df):
    return movie_df.iloc[id_-1][select_col]

# Dataset

In [5]:
# Load Rating Dataset
df = pd.read_csv("./data/u.data", delimiter="\t")
df.columns = ["user_id", "item_id", "rating", "timestamp"]

In [6]:
# Load Movie Info Dataset
movie_df = pd.read_csv('./data/u.item', sep="|", encoding='latin-1', header=None)
movie_df.columns = ['movie id', 'movie title' ,'release date','video release date', 'IMDb URL', 'unknown', 'Action', 
                'Adventure', 'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 
                'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

In [16]:
movie_matrix = movie_df.copy()
movie_matrix.index = movie_matrix["movie id"]
movie_matrix = movie_matrix.drop(["movie id", "movie title", "release date", "video release date", "IMDb URL"], axis=1)
movie_matrix

,unknown,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
movie id,,,,,,,,,,,,,,,,,,,
1,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
4,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1678,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1679,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0
1680,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0


# Modeling

**Collaborative Filtering** is an intuitive recommendation approach based on the idea that **"similar users tend to like similar content."** To measure how similar users are, we compute **cosine similarity** between their rating vectors. The algorithm relies on a **rating matrix**, where each row represents a user, each column represents a movie, and the values are the ratings given by users to movies.

To make a prediction, we **identify users most similar to the target user** and calculate a weighted average of their ratings for a given movie, using their similarity scores as weights. This weighted sum is then **normalized** by dividing it by the sum of the similarities. To recommend movies, we predict ratings for multiple movies and select those with **the highest predicted scores**.

However, **this method has limitations**: it requires users to provide ratings. If a user only watches but **doesn't rate content**, or if the user is new (the **cold start problem**), collaborative filtering **may not work effectively**. In such cases, **content-based filtering** or **hybrid methods** (which combine both approaches) can be used to **make recommendations**.

Video from which I learned about collaborative filtering: [Collaborative Filtering : Data Science Concepts (ritvikmath)](https://youtu.be/Fmtorg_dmM0?si=wHyg-mtl-TUhxCXr)

In [20]:
class SimilarityMatrix:
    def __init__(self):
        self.similarity_matrix = None
    def calculate_matrix(self, rating_matrix):
        prepared = rating_matrix.fillna(0).to_numpy()
        self.similarity_matrix = cosine_similarity(prepared)
        return self.similarity_matrix

In [21]:
%%time
sm = SimilarityMatrix()
similarity_matrix = sm.calculate_matrix(rating_matrix)

CPU times: user 135 ms, sys: 57.8 ms, total: 192 ms
Wall time: 22.1 ms


In [63]:
class ContentBasedFiltering:
    def __init__(self, k=100):
        self.similarity_matrix = None
        self.movie_matrix = None
        self.k = k

    def fit(self, movie_matrix):
        self.similarity_matrix = SimilarityMatrix().calculate_matrix(movie_matrix)
        return self
            
    def recommend(self, movie_id, n=5):
        similarities = self.similarity_matrix[movie_id].copy()
        similarities[movie_id] = -1
        
        top_n_similarities = np.argsort(similarities)[-n:][::-1]
        result = {"MovieId": top_n_similarities, "Similarity Score":  [self.similarity_matrix[movie_id_, movie_id] for movie_id_ in top_n_similarities]}
        result_df = pd.DataFrame(result).sort_values(by=["Similarity Score"], ascending=False).reset_index(drop=True).iloc[:n]
        return result_df

    #def add_user(self, user_ratings: dict):
    #def predict(self, user_id, item_id):


In [64]:
cbf = ContentBasedFiltering()
cbf.fit(movie_matrix)

# Prediction

## Setup Form

In [45]:
movie_ids = train_matrix.columns
movie_titles = [find_movie_by_id(movie_id, "movie title", movie_df) for movie_id in movie_ids]
title2id = dict(zip(movie_titles, movie_ids))

t1_output = widgets.Output()
t1_search = widgets.Text(placeholder='Search...')
t1_dropdown = widgets.Dropdown(options=[])

def update_dropdown(change):
    text = change['new'].lower()
    filtered = [opt for opt in movie_titles if text in opt.lower()][:100]
    t1_dropdown.options = filtered or ['No matches']
    
def t1_func(b):
    movie_title = t1_dropdown.value
    movie_id = title2id[movie_title]
    with t1_output:
        clear_output()
        print(f"Id of \"{movie_title}\" is {movie_id}")
        
t1_search.observe(update_dropdown, names='value')
t1_button = widgets.Button(description="Show Movie Id")
t1_button.on_click(t1_func)
t1 = widgets.VBox([t1_search, t1_dropdown, t1_button, t1_output])

In [69]:
t2_output = widgets.Output()

t2_movie_id_field = widgets.BoundedIntText(
    value=13,
    min=0,
    max=len(movie_matrix.index),
    step=1,
    description='Movie id from dataset:',
    disabled=False,
    style={'description_width': 'initial'}
)

t2_n_movies_field = widgets.IntSlider(
    value=5,
    min=1,
    max=100,
    step=1,
    description='How many movies to recommend:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
def t2_func(b):
    movie_id = t2_movie_id_field.value
    topNmovies = t2_n_movies_field.value
    recommendations = cbf.recommend(movie_id, n=topNmovies)
    recommendations["Movie Title"] = recommendations["MovieId"].apply(lambda movie_id_: find_movie_by_id(movie_id_, "movie title", movie_df))
    recommendations.index = recommendations.index+1
    with t2_output:
        clear_output()
        print(f"Simmilar Movies to \"{find_movie_by_id(movie_id, "movie title", movie_df)}\":")
        display(recommendations[["Movie Title", "Similarity Score"]])

instructions = widgets.HTML("<h3>Enter Movie Id to recommend simmilar movies</h3>", layout=widgets.Layout(width='700px'))
t2_button = widgets.Button(description="Get Recommendations")
t2_button.on_click(t2_func)
t2 = widgets.VBox([instructions, t2_movie_id_field, t2_n_movies_field, t2_button, t2_output])

In [74]:
form = widgets.Tab()
form.children = [t1, t2]
form.set_title(0, 'Search for Movie Id')
form.set_title(1, 'Get Recommendations')

## Prediction Form

In [75]:
display(form)